# Kaggle Remote Tuning Notebook

Use this notebook to benchmark solution profiles on Kaggle compute, pick the best train proxy, generate a submission, and validate it before upload.

In [1]:
from pathlib import Path

URL_FILE = Path('kaggle_remote_url.txt')
if URL_FILE.exists():
    kaggle_url = URL_FILE.read_text(encoding='utf-8').strip()
    print('Kaggle remote URL loaded from', URL_FILE)
    print((kaggle_url[:110] + '...') if len(kaggle_url) > 110 else kaggle_url)
else:
    print('URL file not found:', URL_FILE)

URL file not found: kaggle_remote_url.txt


In [5]:
from pathlib import Path
import os
print('cwd=', Path.cwd())
print('/kaggle/input exists=', Path('/kaggle/input').exists())
if Path('/kaggle/input').exists():
    print('datasets=', sorted([p.name for p in Path('/kaggle/input').iterdir() if p.is_dir()]))
print('solution.py exists in cwd=', Path('solution.py').exists())
print('kaggle_pathfinder.py exists in cwd=', Path('kaggle_pathfinder.py').exists())

cwd= /kaggle/working
/kaggle/input exists= True
datasets= ['datasets']
solution.py exists in cwd= False
kaggle_pathfinder.py exists in cwd= False


In [6]:
from pathlib import Path
hits = []
for root in [Path('/kaggle/input')]:
    if not root.exists():
        continue
    for p in root.rglob('*'):
        if p.name in {'solution.py', 'kaggle_pathfinder.py', 'train.csv', 'test.csv'}:
            hits.append(str(p))
print('found', len(hits), 'interesting files')
for x in hits[:120]:
    print(x)

found 2 interesting files
/kaggle/input/datasets/soufianelasfar/public/train.csv
/kaggle/input/datasets/soufianelasfar/public/test.csv


In [7]:
import urllib.request
try:
    with urllib.request.urlopen('https://example.com', timeout=8) as r:
        print('internet_ok', r.status)
except Exception as e:
    print('internet_blocked', type(e).__name__, str(e)[:180])

internet_ok 200


In [19]:
from pathlib import Path
import urllib.request

FILES = {
    'solution.py': 'https://paste.rs/pBmYc',
    'kaggle_pathfinder.py': 'https://paste.rs/1x3kS',
}

for name, url in FILES.items():
    text = urllib.request.urlopen(url, timeout=30).read().decode('utf-8')
    Path(name).write_text(text, encoding='utf-8')
    print('wrote', name, 'chars=', len(text))

print('solution_exists', Path('solution.py').exists())
print('pathfinder_exists', Path('kaggle_pathfinder.py').exists())

wrote solution.py chars= 59116
wrote kaggle_pathfinder.py chars= 3326
solution_exists True
pathfinder_exists True


In [20]:
import subprocess, sys
print('Running kaggle_pathfinder.py ...')
res = subprocess.run([sys.executable, 'kaggle_pathfinder.py'], capture_output=True, text=True)
print(res.stdout)
if res.stderr.strip():
    print('stderr:', res.stderr)
print('exit_code', res.returncode)

Running kaggle_pathfinder.py ...
=== Kaggle Pathfinder ===
cwd: /kaggle/working
kaggle_input_exists: True
kaggle_input_dirs: ['datasets']

Best match:
DATA_DIR=/kaggle/input/datasets/soufianelasfar/public
Use with solution.py: --data-dir /kaggle/input/datasets/soufianelasfar/public

All matches:
1. {"path": "/kaggle/input/datasets/soufianelasfar/public", "has_required": true, "has_optional": ["sample_submission.csv"], "files": ["sample_submission.csv", "test.csv", "train.csv"]}

exit_code 0


In [16]:
import subprocess
import numpy as np
from xgboost import XGBClassifier

print(subprocess.check_output(['nvidia-smi', '--query-gpu=name,utilization.gpu,memory.used,memory.total', '--format=csv,noheader'], text=True))

# Quick XGBoost-CUDA smoke test (uses GPU path used by solution.py reranker).
X = np.random.randn(4096, 20).astype('float32')
y = (np.random.rand(4096) > 0.7).astype('int32')
model = XGBClassifier(
    n_estimators=30,
    max_depth=4,
    learning_rate=0.1,
    objective='binary:logistic',
    eval_metric='logloss',
    tree_method='hist',
    device='cuda',
    n_jobs=4,
    random_state=42,
    verbosity=0,
    subsample=0.9,
    colsample_bytree=0.9,
    reg_lambda=1.0,
 )
model.fit(X, y)
probs = model.predict_proba(X[:5])[:, 1]
print('xgb_cuda_smoke_ok', probs.tolist())
print(subprocess.check_output(['nvidia-smi', '--query-gpu=name,utilization.gpu,memory.used,memory.total', '--format=csv,noheader'], text=True))

Tesla P100-PCIE-16GB, 0 %, 323 MiB, 16384 MiB

xgb_cuda_smoke_ok [0.2889368534088135, 0.3129478096961975, 0.254325270652771, 0.33364400267601013, 0.2549281120300293]
Tesla P100-PCIE-16GB, 13 %, 355 MiB, 16384 MiB



In [21]:
import subprocess, sys, time
from pathlib import Path

print('Running solution.py with no args ...')
print('gpu_before:', subprocess.check_output(['nvidia-smi', '--query-gpu=utilization.gpu,memory.used,memory.total', '--format=csv,noheader'], text=True).strip())
t0 = time.time()
proc = subprocess.run([sys.executable, 'solution.py'])
dt = time.time() - t0
print('exit_code', proc.returncode, 'elapsed_sec', round(dt, 2))
print('gpu_after:', subprocess.check_output(['nvidia-smi', '--query-gpu=utilization.gpu,memory.used,memory.total', '--format=csv,noheader'], text=True).strip())

out_candidates = [Path('/kaggle/working/submission.csv'), Path('/kaggle/working/working/submission.csv')]
for p in out_candidates:
    print(str(p), 'exists=', p.exists(), 'size=', p.stat().st_size if p.exists() else -1)

Running solution.py with no args ...
gpu_before: 0 %, 355 MiB, 16384 MiB
[info] gpu=Tesla P100-PCIE-16GB, 16384 MiB
[info] data_dir=/kaggle/input/datasets/soufianelasfar/public
[info] loading sampled train rows (sample=1200, total=2766)...
[info] loaded train rows for fitting/tuning: 1200
[info] ranking train chunks...
[train-rank] 1200/1200 (365.1 rows/s)
[reranker] trained xgboost-cuda reranker | samples=28591 aug=7804 pos_rate=0.3118
[info] applying learned reranker to train rankings...
[info] tuning params on train sample...
[tune] best params: Params(keep_ratio=0.7, no_chunk_threshold=0.0, k_default=2, k_numeric=3, k_date=2, k_person=2, k_location=2, redundancy_jaccard=0.78) | train score= 0.412868 f1= 0.093134 eff= 0.732601
[info] output=/kaggle/working/submission.csv
[info] streaming test prediction for 923 rows...
[done] wrote 923 rows to /kaggle/working/submission.csv
exit_code 0 elapsed_sec 73.37
gpu_after: 0 %, 355 MiB, 16384 MiB
/kaggle/working/submission.csv exists= True s

In [22]:
import csv, re
from pathlib import Path

CHUNK_MARKER_RE = re.compile(r'^\[(C\d+)\]\s*', re.MULTILINE)

def chunk_ids(context: str):
    return {m.group(1) for m in CHUNK_MARKER_RE.finditer(context)}

test_path = Path('/kaggle/input/datasets/soufianelasfar/public/test.csv')
sub_path = Path('/kaggle/working/submission.csv')

with test_path.open(newline='', encoding='utf-8') as f:
    test_rows = list(csv.DictReader(f))
with sub_path.open(newline='', encoding='utf-8') as f:
    sub_rows = list(csv.DictReader(f))

valid_by_id = {r['id']: chunk_ids(r['context']) for r in test_rows}
invalid = []
for r in sub_rows:
    ids = [x.strip() for x in (r.get('evidence_ids') or '').split(',') if x.strip()]
    for cid in ids:
        if cid not in valid_by_id.get(r['id'], set()):
            invalid.append((r['id'], cid))

print('columns', list(sub_rows[0].keys()) if sub_rows else [])
print('rows', len(sub_rows), 'expected', len(test_rows))
print('missing', len(set(x['id'] for x in test_rows) - set(x['id'] for x in sub_rows)))
print('extra', len(set(x['id'] for x in sub_rows) - set(x['id'] for x in test_rows)))
print('dupes', len(sub_rows) - len(set(x['id'] for x in sub_rows)))
print('invalid_evidence_ids', len(invalid))
if invalid:
    print('invalid_examples', invalid[:5])

columns ['id', 'evidence_ids', 'answer']
rows 923 expected 923
missing 0
extra 0
dupes 0
invalid_evidence_ids 0


## 1) Environment and Data Paths
This cell auto-detects train/test CSV in either local workspace or Kaggle input.

In [3]:
import os
import csv
import re
import importlib
from pathlib import Path
import pandas as pd

def resolve_project_root(start: Path) -> Path:
    candidates = [
        start,
        Path('/home/nothing/shipd_problems/Cost Efficient RAG Optimization'),
        start / 'shipd_problems' / 'Cost Efficient RAG Optimization',
    ]
    for c in candidates:
        if (c / 'solution.py').exists():
            return c

    search_roots = [start, Path('/home/nothing/shipd_problems')]
    seen = set()
    for base in search_roots:
        base = base.resolve()
        if base in seen or not base.exists():
            continue
        seen.add(base)
        for p in base.rglob('solution.py'):
            parent = p.parent
            if (parent / 'dataset' / 'public' / 'train.csv').exists() and (parent / 'dataset' / 'public' / 'test.csv').exists():
                return parent

    raise FileNotFoundError('Could not locate project root containing solution.py and dataset/public')

ROOT = resolve_project_root(Path.cwd())
os.chdir(ROOT)
SOLUTION_PATH = ROOT / 'solution.py'
if not SOLUTION_PATH.exists():
    raise FileNotFoundError('solution.py must exist in the notebook working directory')

def find_data_dir(root: Path) -> Path:
    local = root / 'dataset' / 'public'
    if (local / 'train.csv').exists() and (local / 'test.csv').exists():
        return local

    alt = root / 'data'
    if (alt / 'train.csv').exists() and (alt / 'test.csv').exists():
        return alt

    kaggle_input = Path('/kaggle/input')
    if kaggle_input.exists():
        for train_path in kaggle_input.rglob('train.csv'):
            parent = train_path.parent
            if (parent / 'test.csv').exists():
                return parent

    raise FileNotFoundError('Could not find train.csv and test.csv in local or /kaggle/input')

DATA_DIR = find_data_dir(ROOT)
OUT_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else (ROOT / 'working')
OUT_DIR.mkdir(parents=True, exist_ok=True)

print('Root:', ROOT)
print('Data dir:', DATA_DIR)
print('Output dir:', OUT_DIR)

FileNotFoundError: Could not locate project root containing solution.py and dataset/public

## 2) Load Solution Module and Build Base Caches

In [ ]:
import solution as sol
importlib.reload(sol)

TRAIN_ROWS = sol.load_rows(DATA_DIR / 'train.csv', has_labels=True)
TEST_ROWS = sol.load_rows(DATA_DIR / 'test.csv', has_labels=False)
BASE_TRAIN_RANKED = sol.build_rankings(TRAIN_ROWS)
BASE_TEST_RANKED = sol.build_rankings(TEST_ROWS)

print('solution module:', Path(sol.__file__).resolve())
print('train rows:', len(TRAIN_ROWS))
print('test rows:', len(TEST_ROWS))

## 3) Define Evaluation and Submission Helpers

In [ ]:
DEFAULT_NO_TUNE_PARAMS = sol.Params(
    keep_ratio=0.62,
    no_chunk_threshold=0.03,
    k_default=3,
    k_numeric=4,
    k_date=3,
    k_person=2,
    k_location=2,
    redundancy_jaccard=0.84,
)

def prepare_rankings(disable_reranker: bool, seed: int = 42):
    train_ranked = BASE_TRAIN_RANKED
    test_ranked = BASE_TEST_RANKED

    if not disable_reranker:
        reranker = sol.train_chunk_reranker(TRAIN_ROWS, train_ranked, seed=seed)
        if reranker is not None:
            train_ranked = sol.apply_chunk_reranker(TRAIN_ROWS, train_ranked, reranker)
            test_ranked = sol.apply_chunk_reranker(TEST_ROWS, test_ranked, reranker)
    return train_ranked, test_ranked

def evaluate_profile(name: str, disable_reranker: bool, qa_model: str, no_tune: bool, seed: int = 42):
    train_ranked, _ = prepare_rankings(disable_reranker=disable_reranker, seed=seed)

    if no_tune:
        params = DEFAULT_NO_TUNE_PARAMS
    else:
        sol.GLOBAL_QA_READER = None
        params = sol.tune_params(TRAIN_ROWS, train_ranked, seed=seed)

    sol.GLOBAL_QA_READER = sol.maybe_init_qa_reader(qa_model)
    metrics = sol.evaluate_on_train(TRAIN_ROWS, train_ranked, params)
    sol.GLOBAL_QA_READER = None

    return {
        'profile': name,
        'disable_reranker': disable_reranker,
        'qa_model': qa_model,
        'no_tune': no_tune,
        'score': metrics['score'],
        'answer_f1': metrics['answer_f1'],
        'evidence_eff': metrics['evidence_eff'],
        'params': params,
    }

def build_submission(profile: dict, output_name: str = 'submission_best.csv', seed: int = 42):
    train_ranked, test_ranked = prepare_rankings(disable_reranker=profile['disable_reranker'], seed=seed)

    if profile['no_tune']:
        params = DEFAULT_NO_TUNE_PARAMS
    else:
        sol.GLOBAL_QA_READER = None
        params = sol.tune_params(TRAIN_ROWS, train_ranked, seed=seed)

    sol.GLOBAL_QA_READER = sol.maybe_init_qa_reader(profile['qa_model'])
    submission_rows = sol.predict_rows(TEST_ROWS, test_ranked, params)
    sol.GLOBAL_QA_READER = None

    output_path = OUT_DIR / output_name
    sol.write_submission(submission_rows, output_path)
    return output_path, params

## 4) Run Profile Sweep
Start with fast profiles first. Run QA profiles only when needed because they are slower.

In [ ]:
profiles = [
    {'name': 'tuned_reranker_no_qa', 'disable_reranker': False, 'qa_model': 'none', 'no_tune': False},
    {'name': 'tuned_no_reranker_no_qa', 'disable_reranker': True, 'qa_model': 'none', 'no_tune': False},
    {'name': 'no_tune_reranker_no_qa', 'disable_reranker': False, 'qa_model': 'none', 'no_tune': True},
    # Uncomment when you want the slow QA-assisted train evaluation:
    # {'name': 'tuned_reranker_tinyqa', 'disable_reranker': False, 'qa_model': 'deepset/tinyroberta-squad2', 'no_tune': False},
]

results = []
for p in profiles:
    print('\n===', p['name'], '===')
    res = evaluate_profile(
        name=p['name'],
        disable_reranker=p['disable_reranker'],
        qa_model=p['qa_model'],
        no_tune=p['no_tune'],
    )
    results.append(res)

results_df = pd.DataFrame([{k: v for k, v in r.items() if k != 'params'} for r in results])
results_df = results_df.sort_values('score', ascending=False).reset_index(drop=True)
display(results_df)

best_name = results_df.iloc[0]['profile']
best_profile = next(p for p in profiles if p['name'] == best_name)
print('Best profile:', best_profile)

## 5) Build Submission with Best Profile

In [ ]:
output_path, selected_params = build_submission(best_profile, output_name='submission_best.csv')
print('Submission written to:', output_path)
print('Selected params:', selected_params)

## 6) Validate Submission Integrity

In [ ]:
with output_path.open(newline='', encoding='utf-8') as f:
    sub_rows = list(csv.DictReader(f))

sub_cols = list(sub_rows[0].keys()) if sub_rows else []
test_ids = [r.rid for r in TEST_ROWS]
sub_ids = [r['id'] for r in sub_rows]

missing_ids = sorted(set(test_ids) - set(sub_ids))
extra_ids = sorted(set(sub_ids) - set(test_ids))
duplicate_ids = len(sub_ids) - len(set(sub_ids))

valid_by_id = {r.rid: {c.cid for c in r.chunks} for r in TEST_ROWS}
invalid_evidence = []
for row in sub_rows:
    rid = row['id']
    ev = [x.strip() for x in (row.get('evidence_ids') or '').split(',') if x.strip()]
    valid = valid_by_id.get(rid, set())
    for cid in ev:
        if cid not in valid:
            invalid_evidence.append((rid, cid))

print('columns:', sub_cols)
print('row_count:', len(sub_rows), 'expected:', len(TEST_ROWS))
print('missing_ids:', len(missing_ids))
print('extra_ids:', len(extra_ids))
print('duplicate_ids:', duplicate_ids)
print('invalid_evidence_ids:', len(invalid_evidence))
if invalid_evidence:
    print('invalid examples:', invalid_evidence[:5])

## 7) Final Export
If validation is clean, copy or rename output_path to /kaggle/working/submission.csv before Kaggle submission.

In [ ]:
from pathlib import Path
import subprocess
import sys

print("Running solution.py...")
cmd = [sys.executable, "solution.py"]
proc = subprocess.run(cmd, cwd=str(Path.cwd()))
print("Exit code:", proc.returncode)

sub_path = Path("working/submission.csv")
print("Submission exists:", sub_path.exists())
if sub_path.exists():
    print("Submission path:", sub_path.resolve())
    print("Submission size bytes:", sub_path.stat().st_size)


: 